# KnobNet - Transfer Experiment: N-sample Learning Curve

**실험 목적**: Black VST에서 학습한 Joint 모델(MERT+HEAD)을 새로운 VST에 이식하면
데이터가 적어도 Scratch보다 빠르게 수렴하는가?

**두 조건 비교**:
- **Transfer**: `joint_full_ft_best.pt` (MERT+HEAD 전체 로드) → 새 VST 데이터로 fine-tune
- **Scratch**: HuggingFace MERT + 랜덤 HEAD → 새 VST 데이터로 처음부터 학습

**N_LIST**: [500, 1000, 2000, 5000, 10000, 20000] (샘플 수 기준)

**공통 설정**:
- MERT unfrozen from start, HEAD lr=1e-3, MERT lr=1e-5
- MAX_EPOCHS=40, PATIENCE=10 (early stop)
- val set: 전체 60k 중 20% = 12k 고정 (seed=42)
- train pool: 나머지 80% = 48k, N개 랜덤 샘플링

**순서**: 환경 설치 → Drive 마운트 → 데이터 해제 → 학습 루프 → 결과 시각화

## 1. 환경 설치

In [ ]:
!pip install -q transformers soundfile torchaudio

## 2. GitHub 클론

In [ ]:
import os

GITHUB_REPO = "https://github.com/kuhberaBubo/KnobNet.git"  # <-- 수정
PROJECT_DIR = "/content/KnobNet"

if not os.path.exists(PROJECT_DIR):
    !git clone {GITHUB_REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print("현재 디렉터리:", os.getcwd())

## 3. Google Drive 마운트 & 데이터 압축 해제

**새 VST 데이터**를 Drive에서 압축 해제합니다. (Black VST 데이터와 별도)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import re
from pathlib import Path

ZIP_FILES = [
    "/content/drive/MyDrive/Colab/dataset/NEW_VST.zip",  # <-- 새 VST 데이터 zip으로 수정
]

DATA_DIR = Path(PROJECT_DIR) / "data"
DATA_DIR.mkdir(exist_ok=True)

!apt-get install -q p7zip-full

for zip_file in ZIP_FILES:
    zip_path = Path(zip_file)
    if re.fullmatch(r'\.z\d+', zip_path.suffix, re.IGNORECASE):
        print(f"[skip] 연속 파트: {zip_path.name}")
        continue
    if not zip_path.exists():
        print(f"[SKIP] 파일 없음: {zip_path}")
        continue
    print(f"압축 해제 중: {zip_path.name} ...")
    !7z x "{zip_file}" -o"{DATA_DIR}" -y
    print("완료")

print("\n데이터 디렉터리 구조:")
!find {DATA_DIR} -maxdepth 3 -type d

## 4. 데이터 확인

In [ ]:
from pathlib import Path

DATA_DIR = Path(PROJECT_DIR) / "data"

print("=== data/ 하위 전체 디렉터리 ===")
!find {DATA_DIR} -type d | sort

print("\n=== samples.csv 위치 ===")
!find {DATA_DIR} -name "samples.csv" | sort

print("\n=== wav 파일 수 ===")
!find {DATA_DIR} -name "*.wav" | wc -l

## 5. 전체 Loader 생성 (고정 val set + train pool)

- `val_split=0.2, seed=42` → 항상 동일한 12k val set 보장
- `train_indices`, `train_ds_base`를 뽑아둬서 이후 N-sample 서브셋에 재사용

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from dataset.loader import make_loaders

WET_DIR = "data/NEW_VST/output"  # <-- 새 VST wet 데이터 경로로 수정

full_train_loader, val_loader = make_loaders(
    dataset_root = PROJECT_DIR,
    wet_dir      = WET_DIR,
    batch_size   = 16,
    val_split    = 0.2,
    num_workers  = 2,
    seed         = 42,
)

# train pool 인덱스 + 기반 dataset 추출
train_indices = list(full_train_loader.dataset.indices)  # Subset.indices
train_ds_base = full_train_loader.dataset.dataset        # KnobDataset (augment=True)

print(f"train pool: {len(train_indices):,}개")
print(f"val set  : {len(val_loader.dataset):,}개  (고정)")

## 6. N-sample 서브셋 로더 헬퍼

In [ ]:
import random
import torch
from torch.utils.data import DataLoader, Subset


def make_n_loader(base_dataset, pool_indices, n, seed=42, batch_size=16):
    """pool_indices 중 n개를 랜덤 샘플링해 DataLoader를 반환."""
    rng = random.Random(seed)
    sampled = rng.sample(pool_indices, min(n, len(pool_indices)))
    return DataLoader(
        Subset(base_dataset, sampled),
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )


print("make_n_loader 정의 완료")

## 7. 실험 Config

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path

from model.model import KnobNetJoint
from utils.config import KNOB_PARAMS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ── 실험 파라미터 ───────────────────────────────────────────────────────────────
N_LIST      = [500, 1000, 2000, 5000, 10000, 20000]
CONDITIONS  = ["transfer", "scratch"]
MAX_EPOCHS  = 40
PATIENCE    = 10
BATCH_SIZE  = 16
N_SEED      = 42   # N 서브샘플링 시드

# ── 경로 ───────────────────────────────────────────────────────────────────────
# Transfer의 소스: Black VST Joint 학습 결과 (Drive에 저장됨)
SOURCE_CKPT = Path("/content/drive/MyDrive/KnobNet/ablation/joint_full_ft/joint_full_ft_best.pt")

# Transfer 실험 결과 저장 위치  (<-- VST 이름에 맞게 수정)
CKPT_DIR    = Path("/content/drive/MyDrive/KnobNet/ablation/transfer")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"N_LIST     : {N_LIST}")
print(f"SOURCE_CKPT: {SOURCE_CKPT}")
print(f"CKPT_DIR   : {CKPT_DIR}")
print(f"SOURCE 존재: {SOURCE_CKPT.exists()}")

## 8. 메인 학습 루프

각 (N, condition) 조합에 대해:
1. N개 샘플 로더 생성
2. 모델 초기화 (Transfer: source 체크포인트 로드 / Scratch: HuggingFace MERT)
3. MAX_EPOCHS=40, PATIENCE=10으로 학습
4. per-epoch 로그 CSV 저장, best 체크포인트 저장

총 12 runs (6 N × 2 조건). 약 2–3시간 소요 예상 (A100 기준).

In [ ]:
from train.train import (
    run_epoch, evaluate_all,
    make_optimizer, save_checkpoint, load_checkpoint,
    log_epoch, log_param_mae, log_accuracy, log_csv,
)

criterion = nn.L1Loss()

# (N, condition) → best val_loss 기록 (summary용)
results = {}

for n in N_LIST:
    for condition in CONDITIONS:
        run_label = f"n{n:05d}_{condition}"
        ckpt_path = CKPT_DIR / f"{run_label}_best.pt"
        csv_path  = CKPT_DIR / f"{run_label}_log.csv"

        print(f"\n{'='*60}")
        print(f"  N={n:,}  /  condition={condition}")
        print(f"{'='*60}")

        # ── N-sample 로더 ──
        train_loader_n = make_n_loader(train_ds_base, train_indices, n,
                                       seed=N_SEED, batch_size=BATCH_SIZE)
        print(f"train samples: {len(train_loader_n.dataset):,}")

        # ── 모델 초기화 ──
        model = KnobNetJoint(
            num_knobs  = len(KNOB_PARAMS),
            freeze_mert= False,
            layer_idx  = 4,
        ).to(device)

        if condition == "transfer":
            load_checkpoint(SOURCE_CKPT, model)  # MERT + HEAD 가중치 로드
            print(f"Transfer: 소스 체크포인트 로드 완료 → {SOURCE_CKPT.name}")
        else:
            print("Scratch: HuggingFace MERT + 랜덤 HEAD")

        # ── 옵티마이저 / 스케줄러 ──
        optimizer = make_optimizer(model, lr=1e-3, phase=2, mert_lr_scale=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=MAX_EPOCHS
        )
        scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

        # ── 기존 CSV 초기화 ──
        if csv_path.exists():
            csv_path.unlink()

        best_val   = float("inf")
        no_improve = 0

        # ── 학습 루프 ──
        for epoch in range(1, MAX_EPOCHS + 1):
            train_loss = run_epoch(model, train_loader_n, optimizer, criterion,
                                   device, scaler)
            metrics    = evaluate_all(model, val_loader, criterion, device,
                                      tolerance=0.1)
            val_loss   = metrics["val_loss"]
            scheduler.step()

            improved = val_loss < best_val
            if improved:
                best_val   = val_loss
                no_improve = 0
                save_checkpoint(ckpt_path, model, epoch, phase=2,
                                val_loss=val_loss,
                                optimizer=optimizer, scheduler=scheduler)
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    print(f"Early stop at epoch {epoch} (patience={PATIENCE})")
                    break

            log_epoch(2, epoch, MAX_EPOCHS, train_loss, val_loss, improved)
            log_param_mae(metrics["mae"])
            log_accuracy(metrics["acc"], tolerance=0.1)
            log_csv(csv_path, 2, epoch, train_loss, val_loss,
                    metrics["mae"], metrics["acc"])

        results[(n, condition)] = {
            "best_val_loss": best_val,
            "mae"          : metrics["mae"],
            "acc"          : metrics["acc"],
        }
        print(f"  완료 → best val_loss: {best_val:.4f}")

print("\n모든 실험 완료")

## 9. Summary CSV 생성

In [ ]:
import csv

summary_path = CKPT_DIR / "summary.csv"
knob_names   = list(results[next(iter(results))]["mae"].keys())

fieldnames = (
    ["n", "condition", "best_val_loss"]
    + [f"mae_{k}" for k in knob_names]
    + [f"acc_{k}" for k in knob_names + ["all"]]
)

with open(summary_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for (n, condition), res in sorted(results.items()):
        row = {
            "n"            : n,
            "condition"    : condition,
            "best_val_loss": f"{res['best_val_loss']:.6f}",
        }
        for k in knob_names:
            row[f"mae_{k}"] = f"{res['mae'].get(k, float('nan')):.6f}"
        for k in knob_names + ["all"]:
            row[f"acc_{k}"] = f"{res['acc'].get(k, float('nan')):.4f}"
        writer.writerow(row)

print(f"Summary 저장: {summary_path}")

# 결과 출력
print(f"\n{'N':>8}  {'condition':<12}  {'val_loss':>9}  {'acc_all':>8}")
print("-" * 48)
for (n, condition), res in sorted(results.items()):
    acc_all = res['acc'].get('all', float('nan'))
    print(f"{n:>8,}  {condition:<12}  {res['best_val_loss']:>9.4f}  {acc_all:>7.1%}")

## 10. Learning Curve 시각화

x축: N (학습 샘플 수, log scale)  
y축 (좌): best val loss  
y축 (우): best Acc_all (±0.1 기준)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

colors = {"transfer": "#2196F3", "scratch": "#FF5722"}
markers = {"transfer": "o", "scratch": "s"}

for condition in CONDITIONS:
    ns     = sorted([n for (n, c) in results if c == condition])
    losses = [results[(n, condition)]["best_val_loss"] for n in ns]
    accs   = [results[(n, condition)]["acc"].get("all", 0.0) for n in ns]

    label = "Transfer (Joint → New VST)" if condition == "transfer" else "Scratch"
    ax1.plot(ns, losses, marker=markers[condition], color=colors[condition],
             label=label, linewidth=2, markersize=7)
    ax2.plot(ns, [a * 100 for a in accs], marker=markers[condition],
             color=colors[condition], label=label, linewidth=2, markersize=7)

ax1.set_xscale("log")
ax1.set_xlabel("N (학습 샘플 수)", fontsize=12)
ax1.set_ylabel("Best Val Loss (MAE)", fontsize=12)
ax1.set_title("Val Loss vs N", fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(N_LIST)
ax1.set_xticklabels([str(n) for n in N_LIST], rotation=30)

ax2.set_xscale("log")
ax2.set_xlabel("N (학습 샘플 수)", fontsize=12)
ax2.set_ylabel("Acc_all (%) [tolerance ±0.1]", fontsize=12)
ax2.set_title("Accuracy vs N", fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xticks(N_LIST)
ax2.set_xticklabels([str(n) for n in N_LIST], rotation=30)

plt.tight_layout()
fig_path = CKPT_DIR / "learning_curve.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {fig_path}")

## (선택) 11. 특정 N/condition 예측 결과 확인

In [ ]:
from train.train import print_predictions

# 확인할 run 설정  (<-- 원하는 N / condition으로 수정)
INSPECT_N         = 5000
INSPECT_CONDITION = "transfer"

inspect_label = f"n{INSPECT_N:05d}_{INSPECT_CONDITION}"
inspect_ckpt  = CKPT_DIR / f"{inspect_label}_best.pt"

model_inspect = KnobNetJoint(
    num_knobs=len(KNOB_PARAMS), freeze_mert=False, layer_idx=4
).to(device)
load_checkpoint(inspect_ckpt, model_inspect)

print(f"\n[{inspect_label}] 예측 결과 (n=8)")
print_predictions(model_inspect, val_loader, device, n=8)